# RAG PDF Question Answering Project

This notebook builds a simple **RAG-based Question Answering system** using:

- LangChain
- PDF document loader
- Hugging Face BGE embeddings **locally** for vector search
- FAISS vector database
- **Ollama local LLM** for answer generation

## Why Ollama?

This version does **not require a Hugging Face API token**.  
The LLM runs locally through Ollama.

## Objective

The objective is to load a PDF document, split it into chunks, convert those chunks into embeddings, store them in FAISS, retrieve the most relevant chunks for a user question, and generate an answer using a local LLM.

## Sample PDF Used

**Health Insurance Coverage Status and Type by Geography: 2021 and 2022**  
Source: U.S. Census Bureau  
File name used in this project: `health-insurance-coverage.pdf`


## 1. Install Required Libraries

Run this cell once if the packages are not installed in your environment.

In [1]:
# Install required packages if needed
# Run this once in VS Code terminal or uncomment and run this cell:
# !pip install langchain langchain-community langchain-core langchain-text-splitters langchain-classic
# !pip install sentence-transformers faiss-cpu pypdf langchain-ollama


## 2. Import Libraries

In [1]:
import torch
print(torch.__version__)

2.12.0+cpu


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_ollama import OllamaLLM

import os
from pathlib import Path


C:\Users\User\AppData\Local\Temp\ipykernel_8844\2784018286.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 3. Load the PDF Document

Place the PDF file in the same folder as this notebook, or update the path below.

In [3]:
pdf_path = r"E:\Rahul Verma\document\D drive\PROJECTS\Vscode\RAG\health-insurance-coverage.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Total PDF pages loaded: {len(documents)}")
print(documents[0].page_content[:500])


Total PDF pages loaded: 18
Health Insurance Coverage Status and Type 
by Geography: 2021 and 2022
American Community Survey Briefs
ACSBR-015
Issued September 2023
Douglas Conway and Breauna Branch
INTRODUCTION
Demographic shifts as well as economic and govern-
ment policy changes can affect people’s access to 
health coverage. For example, between 2021 and 2022, 
the labor market continued to improve, which may 
have affected private coverage in the United States 
during that time.
1 Public policy changes included 
the re


## 4. Split the PDF into Chunks

The PDF content is split into smaller chunks because LLMs cannot efficiently process very large documents at once.

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

final_documents = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(final_documents)}")
print(final_documents[0].page_content[:500])

Total chunks created: 81
Health Insurance Coverage Status and Type 
by Geography: 2021 and 2022
American Community Survey Briefs
ACSBR-015
Issued September 2023
Douglas Conway and Breauna Branch
INTRODUCTION
Demographic shifts as well as economic and govern-
ment policy changes can affect people’s access to 
health coverage. For example, between 2021 and 2022, 
the labor market continued to improve, which may 
have affected private coverage in the United States 
during that time.
1 Public policy changes included 
the re


## 5. Create Hugging Face Embeddings

Embeddings convert text into numerical vectors so that FAISS can search based on meaning.

In [5]:
huggingface_embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

sample_vector = huggingface_embeddings.embed_query(final_documents[0].page_content)
print(f"Embedding vector length: {len(sample_vector)}")

C:\Users\User\AppData\Local\Temp\ipykernel_8844\3917910998.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_embeddings = HuggingFaceBgeEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding vector length: 384


## 6. Store Chunks in FAISS Vector Database

FAISS stores document vectors and helps retrieve the most relevant chunks for a question.

In [6]:
vectorstore = FAISS.from_documents(final_documents, huggingface_embeddings)
print("FAISS vector store created successfully.")

FAISS vector store created successfully.


## 7. Create Retriever


The retriever fetches the top matching chunks for each user question.

In [8]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(retriever)

tags=['FAISS', 'HuggingFaceBgeEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000187BF442F10> search_kwargs={'k': 3}


## 8. Configure Ollama Local LLM

This notebook uses **Ollama** instead of Hugging Face hosted API, so no Hugging Face token is required.

Before running the next code cell:

1. Install Ollama from: `https://ollama.com/download`
2. Open Command Prompt and run one of these:

```bash
ollama pull phi3
```

or

```bash
ollama pull llama3.2
```

Recommended for normal laptop: `phi3`.

Keep Ollama running in the background.


In [11]:
pip install langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [12]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="phi3")

response = llm.invoke("What is health insurance coverage?")
print(response)

Health insurance coverage refers to the protection provided by a specific type of policy that helps in paying for medical expenses. When an individual has this form of insurance, they are covered when receiving care within the network specified by their plan or through any provider outside that with higher costs which may not be fully reimbenered by the insurer but offered as part of a "benefit" such as lower premiums or deductibles. Essentially, it reduces out-of-pocket expenses related to health care and ensures financial protection against high medical bills that could arise from illnesses or injuries.


## 9. Create RAG Prompt Template

The prompt instructs the LLM to answer only from the retrieved context.

In [13]:
prompt_template = """
You are a helpful assistant answering questions from a PDF document.
Use only the given context to answer the question.
If the answer is not available in the context, say: "The document does not mention this information."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

## 10. Create RetrievalQA Chain

This combines the retriever, prompt, and local Ollama LLM.


In [14]:
retrievalQA = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

print("RetrievalQA chain created successfully.")


RetrievalQA chain created successfully.


## 11. Ask Questions from the PDF

In [15]:
query = "What is health insurance coverage?"

result = retrievalQA.invoke({"query": query})

print("Answer:")
print(result["result"])

print("\nSource Documents:")
for i, source_doc in enumerate(result["source_documents"], start=1):
    print(f"\nSource {i}: Page {source_doc.metadata.get('page')}")
    print(source_doc.page_content[:500])


Answer:
Health insurance coverage, as presented in this context from a brief using data from the American Community Survey, refers to an individual's status with regard to having some form of comprehensive health insurance. This encompasses being covered by any type of private or public health insurance at the time of interview for state-level estimates in 2022. It excludes single-service plans like accident, disability, dental, vision, and prescription medicine plans as they do not cover basic health care needs comprehensively.

Source Documents:

Source 1: Page 1
2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current c

## 12. More Sample Questions

Try these questions:

1. What is health insurance coverage?
2. What was the uninsured rate in 2022?
3. Which state had the lowest uninsured rate in 2022?
4. Which state had the highest uninsured rate in 2022?
5. What is the difference between private and public health insurance?
6. What changed in uninsured rates from 2021 to 2022?

In [ ]:
sample_questions = [
    "What is health insurance coverage?",
    "What was the uninsured rate in 2022?",
    "Which state had the lowest uninsured rate in 2022?",
    "Which state had the highest uninsured rate in 2022?",
    "What is the difference between private and public health insurance?",
    "What changed in uninsured rates from 2021 to 2022?"
]

for question in sample_questions:
    print("\nQuestion:", question)
    response = retrievalQA.invoke({"query": question})
    print("Answer:", response["result"])
    print("-" * 80)



Question: What is health insurance coverage?
Answer: Health insurance coverage, as defined by this brief based on data from the American Community Survey (ACS), refers to an individual's status regarding comprehensive health care needs being covered either through private or public means. Comprehensive health insurance does not include single-service plans like accident, disability, dental, vision, or prescription medicine plans. The measure of coverage depends on whether the respondent was provided any type of health insurance at the time of interview, including employer/union plans, individually purchased policies from an insurance company or through a marketplace such as healthcare.gov, TRICARE for military members and their dependents, federal programs like Medicare, Medicaid, CHIP, state-specific health plans, CHAMPVA (Civilian Health and Medical Program at the Department of Veterans Affairs), or care provided by the IHS is considered comprehensive. The uninsured are those withou

## Project Summary

This notebook demonstrates a complete beginner-level RAG pipeline:

1. Load PDF
2. Split text into chunks
3. Create embeddings
4. Store vectors in FAISS
5. Retrieve relevant chunks
6. Send context to **Ollama local LLM**
7. Generate answer from document content

This version does not need a Hugging Face API token.

You can replace the sample PDF with your own PDF and ask questions from it.
